In [1]:
# ── Celda 1: Instalación ──────────────────────────────────────────────────────
!pip install -q langchain langchain-mistralai langchain-experimental langchain-community langchain-classic faiss-cpu pandas tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# ── Celda 2: Carga del dataset ────────────────────────────────────────────────
from google.colab import files
import pandas as pd
import io

print("📂 Subí tu archivo 'sales_data_normalizado.csv'")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"✅ Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head(3)

📂 Subí tu archivo 'sales_data_normalizado.csv'


Saving sales_data_normalizado.csv to sales_data_normalizado.csv
✅ Dataset cargado: 2823 filas × 25 columnas


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,-0.522891,0.596978,2,-0.370825,2003-02-24,Shipped,1,2,2003,...,897 Long Airport Avenue,Desconocido,NYC,NY,10022,USA,Desconocido,Yu,Kwai,Small
1,10121,-0.112201,-0.114450,5,-0.427897,2003-05-07,Shipped,2,5,2003,...,59 rue de l'Abbaye,Desconocido,Reims,Desconocido,51100,France,EMEA,Henriot,Paul,Small
2,10134,0.606505,0.549384,2,0.179443,2003-07-01,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,Desconocido,Paris,Desconocido,75508,France,EMEA,Da Cunha,Daniel,Medium


In [3]:
# ── Celda 3: Corpus / contexto del dataset ────────────────────────────────────
import json

numeric_summary = df.describe().round(4).to_string()

categorical_cols = ["STATUS", "PRODUCTLINE", "DEALSIZE", "COUNTRY", "TERRITORY", "YEAR_ID", "QTR_ID"]
cat_summary = {col: df[col].value_counts().to_dict() for col in categorical_cols if col in df.columns}

prefix = f"""
Eres un agente experto en análisis de datos de ventas.
Trabajas con un DataFrame de pandas llamado `df` que contiene datos de órdenes de ventas normalizados.

=== ESTRUCTURA DEL DATASET ===
- Total de filas: {df.shape[0]}
- Total de columnas: {df.shape[1]}
- Columnas: {list(df.columns)}

=== DESCRIPCIÓN DE COLUMNAS ===
- ORDERNUMBER: Identificador único de la orden
- QUANTITYORDERED: Cantidad pedida (normalizada con Z-score)
- PRICEEACH: Precio unitario (normalizado con Z-score)
- ORDERLINENUMBER: Número de línea dentro de la orden
- SALES: Monto de venta (normalizado con Z-score)
- ORDERDATE: Fecha de la orden (formato YYYY-MM-DD)
- STATUS: Estado de la orden — valores: {df['STATUS'].unique().tolist()}
- QTR_ID: Trimestre (1 a 4)
- MONTH_ID: Mes (1 a 12)
- YEAR_ID: Año — valores: {df['YEAR_ID'].unique().tolist()}
- PRODUCTLINE: Línea de producto — valores: {df['PRODUCTLINE'].unique().tolist()}
- MSRP: Precio sugerido de fabricante (normalizado)
- PRODUCTCODE: Código de producto
- CUSTOMERNAME: Nombre del cliente
- PHONE / ADDRESSLINE1 / ADDRESSLINE2 / CITY / STATE / POSTALCODE / COUNTRY: Datos de contacto
- TERRITORY: Territorio comercial — valores: {df['TERRITORY'].unique().tolist()}
- CONTACTLASTNAME / CONTACTFIRSTNAME: Contacto del cliente
- DEALSIZE: Tamaño del trato — valores: {df['DEALSIZE'].unique().tolist()}

=== NOTA SOBRE NORMALIZACIÓN ===
Las columnas QUANTITYORDERED, PRICEEACH, SALES y MSRP están normalizadas con Z-score.
Valores positivos están por encima del promedio; negativos, por debajo.
Mencioná siempre que los valores son Z-scores normalizados.

=== ESTADÍSTICAS DESCRIPTIVAS ===
{numeric_summary}

=== DISTRIBUCIÓN DE VARIABLES CATEGÓRICAS ===
{json.dumps(cat_summary, indent=2, ensure_ascii=False)}

Cuando respondas:
1. Usá siempre el DataFrame `df` para tus cálculos.
2. Respondé en español.
3. Aclará cuando los valores son Z-scores normalizados.
4. Sé conciso pero completo.
5. Si no podés responder, decilo claramente.
"""

print("✅ Contexto del dataset construido correctamente.")
print(f"   Longitud del corpus: {len(prefix)} caracteres")

✅ Contexto del dataset construido correctamente.
   Longitud del corpus: 4376 caracteres


In [4]:
# ── Celda 4: API Key + LLM (Mistral) + Agente Pandas ─────────────────────────
from google.colab import userdata
from langchain_mistralai import ChatMistralAI
from langchain_experimental.agents import create_pandas_dataframe_agent
import os

# API Key desde Colab Secrets (panel 🔑 → MISTRAL_API_KEY)
os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")

# LLM
llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
)

# Crear el agente
agent = create_pandas_dataframe_agent(
    llm,
    df,
    prefix=prefix,
    verbose=True,
    allow_dangerous_code=True,
    handle_parsing_errors=True,
    agent_type="tool-calling"
)

print("✅ Agente creado correctamente")

/tmp/ipykernel_2762/3342492018.py:4: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents import create_pandas_dataframe_agent


✅ Agente creado correctamente


/usr/local/lib/python3.12/dist-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(


In [5]:
# ── Celda 5: Consulta puntual al agente Pandas ───────────────────────────────
def preguntar(pregunta: str):
    print(f"\n{'='*60}")
    print(f"❓ Pregunta: {pregunta}")
    print('='*60)
    respuesta = agent.invoke({"input": pregunta})
    print(f"\n✅ Respuesta:\n{respuesta['output']}")
    return respuesta['output']

# Ejemplo
preguntar("¿Cuántas órdenes hay por cada línea de producto?")


❓ Pregunta: ¿Cuántas órdenes hay por cada línea de producto?


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "orders_by_productline = df.groupby('PRODUCTLINE')['ORDERNUMBER'].nunique().reset_index()\norders_by_productline.columns = ['LÍNEA DE PRODUCTO', 'CANTIDAD DE ÓRDENES']\norders_by_productline.sort_values(by='CANTIDAD DE ÓRDENES', ascending=False)"}`


  LÍNEA DE PRODUCTO  CANTIDAD DE ÓRDENES
0      Classic Cars                  199
6      Vintage Cars                  175
5  Trucks and Buses                   73
1       Motorcycles                   72
3             Ships                   65
2            Planes                   59
4            Trains                   45Aquí está la cantidad de **órdenes únicas** por cada línea de producto:

| **LÍNEA DE PRODUCTO**  | **CANTIDAD DE ÓRDENES** |
|------------------------|-------------------------|
| Classic Cars           | 199                     |
| Vintage Cars           | 175             

'Aquí está la cantidad de **órdenes únicas** por cada línea de producto:\n\n| **LÍNEA DE PRODUCTO**  | **CANTIDAD DE ÓRDENES** |\n|------------------------|-------------------------|\n| Classic Cars           | 199                     |\n| Vintage Cars           | 175                     |\n| Trucks and Buses       | 73                      |\n| Motorcycles            | 72                      |\n| Ships                  | 65                      |\n| Planes                 | 59                      |\n| Trains                 | 45                      |'

In [6]:
# ── Celda 6: Loop interactivo — Agente Pandas ────────────────────────────────
print("🤖 Agente Pandas listo. Escribí 'salir' para terminar.\n")

while True:
    pregunta = input("Vos: ")
    if pregunta.strip().lower() in ["salir", "exit", "quit"]:
        print("👋 Sesión finalizada.")
        break
    if pregunta.strip() == "":
        continue
    preguntar(pregunta)

🤖 Agente Pandas listo. Escribí 'salir' para terminar.

Vos: salir
👋 Sesión finalizada.


---
## 🔍 RAG — Recuperación semántica con FAISS + Mistral

In [7]:
# ── Celda 7: Embeddings con Mistral ──────────────────────────────────────────
from langchain_mistralai import MistralAIEmbeddings

embeddings = MistralAIEmbeddings(
    model="mistral-embed",
)

print("✅ Modelo de embeddings cargado: mistral-embed")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

✅ Modelo de embeddings cargado: mistral-embed


In [8]:
# ── Celda 8: Convertir filas del CSV en documentos ───────────────────────────
from langchain_core.documents import Document

documentos = []

for _, row in df.iterrows():
    contenido = (
        f"Orden: {row['ORDERNUMBER']} | "
        f"Cliente: {row['CUSTOMERNAME']} | "
        f"Producto: {row['PRODUCTLINE']} | "
        f"Código: {row['PRODUCTCODE']} | "
        f"Estado: {row['STATUS']} | "
        f"Deal: {row['DEALSIZE']} | "
        f"País: {row['COUNTRY']} | "
        f"Territorio: {row['TERRITORY']} | "
        f"Año: {row['YEAR_ID']} | "
        f"Trimestre: {row['QTR_ID']} | "
        f"Mes: {row['MONTH_ID']} | "
        f"Ventas (Z): {row['SALES']:.4f} | "
        f"Cantidad (Z): {row['QUANTITYORDERED']:.4f} | "
        f"Precio (Z): {row['PRICEEACH']:.4f}"
    )
    metadata = {
        "order_number": int(row["ORDERNUMBER"]),
        "customer": row["CUSTOMERNAME"],
        "product_line": row["PRODUCTLINE"],
        "status": row["STATUS"],
        "country": row["COUNTRY"],
        "year": int(row["YEAR_ID"]),
        "deal_size": row["DEALSIZE"],
    }
    documentos.append(Document(page_content=contenido, metadata=metadata))

print(f"✅ {len(documentos)} documentos generados desde el CSV")

✅ 2823 documentos generados desde el CSV


In [9]:
# ── Celda 9: Base vectorial con FAISS ────────────────────────────────────────
from langchain_community.vectorstores import FAISS

# Ajustá LIMITE según tu cuota de API (Mistral cobra por embedding)
LIMITE = 500

vectorstore = FAISS.from_documents(
    documents=documentos[:LIMITE],
    embedding=embeddings,
)

print(f"✅ Base vectorial FAISS creada con {LIMITE} documentos")

✅ Base vectorial FAISS creada con 500 documentos


In [10]:
# ── Celda 10: Agente RAG (LLM + Retriever + Prompt) ──────────────────────────
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

# Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

# Prompt
rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
Eres un agente experto en análisis de ventas.
Usás el siguiente contexto recuperado del dataset para responder la pregunta.
Respondé siempre en español. Si no encontrás la respuesta en el contexto, decilo claramente.

RECUERDA: Las columnas SALES, QUANTITYORDERED, PRICEEACH y MSRP están normalizadas con Z-score.

Contexto:
{context}

Pregunta: {question}

Respuesta:"""
)

# Agente RAG
agente_rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": rag_prompt},
)

print("✅ Agente RAG creado correctamente")

✅ Agente RAG creado correctamente


In [ ]:
# ── Celda 11: Loop interactivo — Agente RAG ───────────────────────────────────
print("🔍 Agente RAG listo. Escribí 'salir' para terminar.\n")

while True:
    pregunta = input("Vos: ")
    if pregunta.strip().lower() in ["salir", "exit", "quit"]:
        print("👋 Sesión finalizada.")
        break
    if pregunta.strip() == "":
        continue

    print(f"\n{'='*60}")
    print(f"❓ Pregunta: {pregunta}")
    print('='*60)

    resultado = agente_rag.invoke({"query": pregunta})

    print(f"\n✅ Respuesta:\n{resultado['result']}")

    print(f"\n📄 Fuentes recuperadas ({len(resultado['source_documents'])}):")
    for i, doc in enumerate(resultado['source_documents'], 1):
        print(f"  [{i}] Orden {doc.metadata['order_number']} | "
              f"{doc.metadata['customer']} | "
              f"{doc.metadata['product_line']} | "
              f"{doc.metadata['status']}")
    print()

🔍 Agente RAG listo. Escribí 'salir' para terminar.

Vos: cual es el cliente con mas orden?

❓ Pregunta: cual es el cliente con mas orden?

✅ Respuesta:
En el contexto proporcionado, los clientes con órdenes son:

1. **CAF Imports** (Orden 10231 - 2 registros)
2. **Euro Shopping Channel** (Orden 10424 - 2 registros)
3. **L'ordine Souveniers** (Orden 10266 - 1 registro)

**Respuesta:**
El cliente con **más órdenes** en este contexto es **CAF Imports** y **Euro Shopping Channel**, ambos con **2 órdenes cada uno**. No hay un único cliente con más órdenes que los demás en los datos mostrados.

📄 Fuentes recuperadas (5):
  [1] Orden 10231 | CAF Imports | Classic Cars | Shipped
  [2] Orden 10424 | Euro Shopping Channel | Trucks and Buses | In Process
  [3] Orden 10231 | CAF Imports | Classic Cars | Shipped
  [4] Orden 10266 | L'ordine Souveniers | Classic Cars | Shipped
  [5] Orden 10424 | Euro Shopping Channel | Trucks and Buses | In Process

